# Lab 02 - PySpark Transformation Lab
## Reinsurance Analytics

---

### Learning Objectives

By the end of this lab, you will be able to:

1. **Read** CSV and JSON data into PySpark DataFrames, including explicit schema with `StructType` and nested fields
2. **Filter** rows using multiple conditions combined with `&` and `|`, including `isin()` for `IN` clauses
3. **Transform columns** using conditional logic (`when/otherwise`) and date extraction (`year()`)
4. **Join** DataFrames to enrich datasets and access nested struct fields with dot notation
5. **Aggregate** data using `groupBy` with multiple aggregate functions (`F.sum`, `avg`, `count`)
6. **Apply window functions** including `dense_rank()`, running totals, and `lag()`/`lead()` comparisons
7. **Work with array functions** including `explode()`, `array_contains()`, and `array_distinct()`
8. **Save results** to Parquet files

---

### How This Lab Works

Each exercise follows the same pattern:
- A **SQL query** is shown first
- Then a **PySpark translation** with blanks (`________`) for you to fill in
- Replace each `________` with the correct PySpark code

---

### SQL to PySpark Equivalence

In [1]:
displayHTML("""
<div style="background-color: #F9F7F4; padding: 20px; border-radius: 12px; border: 1px solid #E8E5E0; margin: 10px 0; box-shadow: 0 2px 8px rgba(0,0,0,0.06);">

<table style="width:100%; border-collapse: collapse; font-size: 14px;">
<tr style="background-color: #1B3A4B; color: #FFFFFF;">
  <th style="padding: 12px; text-align: left; width: 25%; border-radius: 6px 0 0 0;">Purpose</th>
  <th style="padding: 12px; text-align: left; width: 35%;">SQL</th>
  <th style="padding: 12px; text-align: left; width: 40%; border-radius: 0 6px 0 0;">PySpark DataFrame API</th>
</tr>
<tr style="background-color: #FFFFFF;">
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><strong>Select columns</strong></td>
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><code>SELECT col1, col2</code></td>
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><code>df.select("col1", "col2")</code></td>
</tr>
<tr style="background-color: #F9F7F4;">
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><strong>Filter rows</strong></td>
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><code>WHERE condition</code></td>
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><code>df.filter(condition)</code></td>
</tr>
<tr style="background-color: #FFFFFF;">
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><strong>Rename column</strong></td>
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><code>col AS alias</code></td>
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><code>col("col").alias("alias")</code></td>
</tr>
<tr style="background-color: #F9F7F4;">
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><strong>Conditional logic</strong></td>
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><code>CASE WHEN ... THEN ... END</code></td>
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><code>when(..., ...).otherwise(...)</code></td>
</tr>
<tr style="background-color: #FFFFFF;">
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><strong>Combine tables</strong></td>
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><code>JOIN ... ON</code></td>
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><code>df1.join(df2, condition, "type")</code></td>
</tr>
<tr style="background-color: #F9F7F4;">
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><strong>Aggregate data</strong></td>
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><code>GROUP BY ... AGG()</code></td>
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><code>df.groupBy(...).agg(...)</code></td>
</tr>
<tr style="background-color: #FFFFFF;">
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><strong>Window function</strong></td>
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><code>OVER (PARTITION BY ... ORDER BY ...)</code></td>
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><code>Window.partitionBy(...).orderBy(...)</code></td>
</tr>
<tr style="background-color: #F9F7F4;">
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><strong>Sort results</strong></td>
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><code>ORDER BY col DESC</code></td>
  <td style="padding: 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><code>df.orderBy(desc("col"))</code></td>
</tr>
<tr style="background-color: #FFFFFF;">
  <td style="padding: 10px; color: #1B1F24;"><strong>Array functions</strong></td>
  <td style="padding: 10px; color: #1B1F24;"><code>LATERAL FLATTEN / EXPLODE</code></td>
  <td style="padding: 10px; color: #1B1F24;"><code>explode(), array_contains(), array_distinct()</code></td>
</tr>
</table>

</div>
""")

Purpose,SQL,PySpark DataFrame API
Select columns,"SELECT col1, col2","df.select(""col1"", ""col2"")"
Filter rows,WHERE condition,df.filter(condition)
Rename column,col AS alias,"col(""col"").alias(""alias"")"
Conditional logic,CASE WHEN ... THEN ... END,"when(..., ...).otherwise(...)"
Combine tables,JOIN ... ON,"df1.join(df2, condition, ""type"")"
Aggregate data,GROUP BY ... AGG(),df.groupBy(...).agg(...)
Window function,OVER (PARTITION BY ... ORDER BY ...),Window.partitionBy(...).orderBy(...)
Sort results,ORDER BY col DESC,"df.orderBy(desc(""col""))"
Array functions,LATERAL FLATTEN / EXPLODE,"explode(), array_contains(), array_distinct()"


---
## Setup

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, lit, when, count, avg,
    concat, concat_ws, year, month, date_format, desc, asc,
    row_number, rank, dense_rank, lag, lead, ntile
)
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType

CATALOG_NAME = "axa_training"  # Change to your catalog name
SCHEMA_ASSETS = "lab_assets"
VOLUME_NAME = "training_volume"
VOLUME_PATH = f"/Volumes/{CATALOG_NAME}/{SCHEMA_ASSETS}/{VOLUME_NAME}"
RAW_DATA_PATH = f"{VOLUME_PATH}/raw_data"
CONTRACTS_PATH = f"{RAW_DATA_PATH}/reinsurance_contracts"
ENTITIES_PATH = f"{RAW_DATA_PATH}/axa_entities"

print("Configuration loaded")
print(f"Contracts Path: {CONTRACTS_PATH}")
print(f"Entities Path: {ENTITIES_PATH}")

---
## Section 1: Reading Data

Before we can transform data, we need to load it into DataFrames. PySpark supports many file formats natively. The two most common patterns are:

- **CSV files**: Use `spark.read.option(...).csv(path)` -- you typically need `header` and `inferSchema` options
- **JSON files**: Use `spark.read.json(path)` -- schema is inferred automatically from the JSON structure

Once loaded, we can also register DataFrames as **temporary views** so we can query them with SQL alongside PySpark.

### Exercise 1.1 - Read CSV (Reinsurance Contracts)

The reinsurance contracts dataset is stored as CSV files with a header row. We need to tell Spark to:
1. Treat the first row as column names (`header`)
2. Automatically detect data types (`inferSchema`)

**Fill in the blanks** to load the CSV data.

In [ ]:
# Exercise 1.1 - Read the reinsurance contracts CSV
# Fill in the blanks to configure the CSV reader

contracts_df = (
    spark.read
    .option("________", "true")
    .option("________", "true")
    .csv(CONTRACTS_PATH)
)

print(f"Contracts loaded: {contracts_df.count()} rows")
contracts_df.printSchema()
display(contracts_df.limit(5))

In [ ]:
# Verification 1.1
assert contracts_df.count() > 0, "DataFrame is empty - check your .csv() path"
assert "contract_id" in contracts_df.columns, "Missing contract_id - check header option"
assert len(contracts_df.columns) > 3, "Too few columns - check inferSchema option"
print(f"Exercise 1.1 passed! {contracts_df.count()} rows, {len(contracts_df.columns)} columns")

In [1]:
displayHTML("""
<details style="margin: 10px 0;">
  <summary style="cursor: pointer; color: #1B3A4B; font-weight: bold; padding: 8px 0;">Click to reveal answer</summary>
  <div style="background: #F9F7F4; padding: 12px 16px; border-radius: 8px; margin-top: 8px; border-left: 4px solid #F9A825;">
    <ul style="margin: 8px 0; padding-left: 20px;">
  <li style="margin: 6px 0; color: #1B1F24;">First blank: <code>"header"</code></li>
  <li style="margin: 6px 0; color: #1B1F24;">Second blank: <code>"inferSchema"</code></li>
</ul>
  </div>
</details>
""")

Click to reveal answer 
 
 
 First blank: "header" 
 Second blank: "inferSchema"

### Exercise 1.2 - Read JSON with Schema and Nested Fields (AXA Entities)

In production, **define an explicit schema** with `StructType` instead of relying on inference. The entities JSON has **nested structs**: `address` (city, postal_code) and `contact` (email, phone).

**Your tasks:**
1. Define a `StructType` schema including nested `StructType` for `address` and `contact`
2. Read the JSON using that schema
3. Access nested fields with dot notation: `col("address.city")`

In [ ]:
# Exercise 1.2 - Read JSON with explicit schema including nested structs

from pyspark.sql.types import StructType, StructField, StringType

# TODO: Fill in the blanks to define the nested schema
entities_schema = StructType([
    StructField("entity_id", StringType(), True),
    StructField("entity_name", StringType(), True),
    StructField("entity_type", StringType(), True),
    StructField("region", StringType(), True),
    StructField("country", StringType(), True),
    StructField("address", ________([                 # Hint: nested struct uses StructType
        StructField("city", StringType(), True),
        StructField("postal_code", StringType(), True),
    ]), True),
    StructField("contact", ________([
        StructField("email", StringType(), True),
        StructField("phone", StringType(), True),
    ]), True),
])

# Read JSON with explicit schema
entities_df = spark.read.schema(entities_schema).json(ENTITIES_PATH)

print(f"Total entities: {entities_df.count()}")
entities_df.printSchema()

# TODO: Access nested fields using dot notation
# Display entity_name, city (from address), and email (from contact)
entities_df.select(
    col("entity_name"),
    col("________").alias("city"),          # Hint: use field.subfield notation
    col("________").alias("email")          # Hint: use field.subfield notation
))

In [ ]:
# Verification 1.2
assert entities_df.count() > 0, "DataFrame is empty"
assert "address" in entities_df.columns, "Missing address struct - check StructType nesting"
assert "contact" in entities_df.columns, "Missing contact struct - check StructType nesting"
city_val = entities_df.select(col("address.city")).first()[0]
assert city_val is not None, "address.city is null - check nested schema"
print(f"Exercise 1.2 passed! {entities_df.count()} entities with nested address.city={city_val}")

In [1]:
displayHTML("""
<details style="margin: 10px 0;">
  <summary style="cursor: pointer; color: #1B3A4B; font-weight: bold; padding: 8px 0;">Click to reveal hints</summary>
  <div style="background: #F9F7F4; padding: 12px 16px; border-radius: 8px; margin-top: 8px; border-left: 4px solid #F9A825;">
    <ul style="margin: 8px 0; padding-left: 20px;">
  <li style="margin: 6px 0; color: #1B1F24;">First two blanks: <code>StructType</code> — a nested struct is just another <code>StructType</code> inside a <code>StructField</code></li>
  <li style="margin: 6px 0; color: #1B1F24;">Third blank: <code>address.city</code> — use <code>field.subfield</code> dot notation to access nested fields</li>
  <li style="margin: 6px 0; color: #1B1F24;">Fourth blank: <code>contact.email</code> — same <code>field.subfield</code> pattern</li>
</ul>
  </div>
</details>
""")

Click to reveal hints 
 
 
 First two blanks: StructType — a nested struct is just another StructType inside a StructField 
 Third blank: address.city — use field.subfield dot notation to access nested fields 
 Fourth blank: contact.email — same field.subfield pattern

### Register Temporary Views for SQL Comparisons

Throughout this lab, we will show the SQL equivalent before each PySpark exercise. To make the SQL examples runnable, we register our DataFrames as temporary views.

In [ ]:
# Register DataFrames as temp views for SQL comparisons
contracts_df.createOrReplaceTempView("contracts")
entities_df.createOrReplaceTempView("entities")

print("Temp views registered: 'contracts', 'entities'")

In [1]:
displayHTML("""
<div style="background-color: #F1F8E9; padding: 14px 18px; border-radius: 10px; margin: 10px 0; border-left: 4px solid #00A972;">
<strong style="color: #1B1F24;">Key Takeaway:</strong> <span style="color: #1B1F24;">PySpark reads multiple file formats natively:</span>
<ul style="margin: 6px 0 0 0; padding-left: 20px; color: #1B1F24;">
  <li><strong>CSV</strong>: <code>spark.read.option("header","true").option("inferSchema","true").csv(path)</code></li>
  <li><strong>JSON</strong>: <code>spark.read.json(path)</code> -- schema inferred automatically</li>
  <li><strong>Parquet</strong>: <code>spark.read.parquet(path)</code> -- schema embedded in file, fastest format</li>
  <li><strong>Delta</strong>: <code>spark.read.format("delta").load(path)</code> -- ACID transactions + time travel</li>
  <li><strong>With explicit schema</strong>: <code>spark.read.schema(my_schema).json(path)</code> -- best practice for production</li>
</ul>
<span style="color: #1B1F24;">All return DataFrames that can be queried with the DataFrame API or registered as temp views for SQL.</span>
</div>
""")

Key Takeaway: PySpark reads multiple file formats natively: 
 
 CSV : spark.read.option("header","true").option("inferSchema","true").csv(path) 
 JSON : spark.read.json(path) -- schema inferred automatically 
 Parquet : spark.read.parquet(path) -- schema embedded in file, fastest format 
 Delta : spark.read.format("delta").load(path) -- ACID transactions + time travel 
 With explicit schema : spark.read.schema(my_schema).json(path) -- best practice for production 
 
 All return DataFrames that can be queried with the DataFrame API or registered as temp views for SQL.

---
## Section 2: Filtering Data

Filtering is one of the most common operations in data analysis. In SQL, you use `WHERE` clauses. In PySpark, the equivalent is `.filter()` or `.where()` (they are identical).

Key syntax differences:
- SQL `=` becomes PySpark `==`
- SQL `AND` becomes PySpark `&` (with parentheses around each condition)
- SQL `OR` becomes PySpark `|`
- SQL `IN (...)` becomes PySpark `.isin(...)`

### Exercise 2.1 - Filter with Multiple Conditions

Find all contracts that are **ACTIVE**, have a **premium above 5,000,000**, and are denominated in **EUR**.

**Fill in the blanks** to build the complete `.filter()` call with three conditions combined using `&`.

**Important:** In PySpark, when combining conditions with `&`, each condition must be wrapped in parentheses.

In [ ]:
%sql
SELECT * FROM contracts
WHERE status = 'ACTIVE'
  AND premium_amount > 5000000
  AND currency = 'EUR'

In [ ]:
# Exercise 2.1 - Filter: ACTIVE contracts with premium > 5M in EUR
# Remember: each condition wrapped in () and combined with &

# Write the full filter expression with 3 conditions:
# status == "ACTIVE", premium_amount > 5000000, currency == "EUR"
# Combined with & and each condition in parentheses
high_value_active = contracts_df.filter(
    ________
)

print(f"High-value active EUR contracts: {high_value_active.count()}")
display(high_value_active.limit(5))

In [ ]:
# Verification 2.1
assert high_value_active.count() > 0, "No results found - check your filter conditions"
assert high_value_active.count() < contracts_df.count(), "Count should be less than total - filter not applied"
# Verify all rows match the conditions
statuses = [r.status for r in high_value_active.select("status").distinct().collect()]
assert statuses == ["ACTIVE"], f"Expected only ACTIVE, got {statuses}"
min_premium = high_value_active.agg(F.min("premium_amount")).first()[0]
assert min_premium > 5000000, f"Found premium {min_premium} <= 5M - check > operator"
currencies = [r.currency for r in high_value_active.select("currency").distinct().collect()]
assert currencies == ["EUR"], f"Expected only EUR, got {currencies}"
print(f"Exercise 2.1 passed! {high_value_active.count()} high-value active EUR contracts")

In [1]:
displayHTML("""
<details style="margin: 10px 0;">
  <summary style="cursor: pointer; color: #1B3A4B; font-weight: bold; padding: 8px 0;">Click to reveal answer</summary>
  <div style="background: #F9F7F4; padding: 12px 16px; border-radius: 8px; margin-top: 8px; border-left: 4px solid #F9A825;">
    <pre style="margin: 8px 0; color: #1B1F24; background: #FFFFFF; padding: 10px; border-radius: 6px;">
high_value_active = contracts_df.filter(
    (col("status") == "ACTIVE") &amp;
    (col("premium_amount") &gt; 5000000) &amp;
    (col("currency") == "EUR")
)
    </pre>
  </div>
</details>
""")

Click to reveal answer 
 
 
high_value_active = contracts_df.filter(
 (col("status") == "ACTIVE") &
 (col("premium_amount") > 5000000) &
 (col("currency") == "EUR")
)

### Exercise 2.2 - Combine Filter, IN Clause, and Column Selection

Find contracts from specific reinsurers (`SWISS-RE-001`, `MUNICH-RE-001`, `HANNOVER-RE-001`) that are **ACTIVE**, then show only the columns `contract_id`, `reinsurer_id`, `premium_amount`, `currency`, and `status`.

**Fill in the blanks** to:
1. Build the `.filter()` combining `.isin()` with a status condition
2. `.select()` only the required columns
3. `.orderBy()` by premium descending

In [ ]:
%sql
SELECT contract_id, reinsurer_id, premium_amount, currency, status
FROM contracts
WHERE reinsurer_id IN ('SWISS-RE-001', 'MUNICH-RE-001', 'HANNOVER-RE-001')
  AND status = 'ACTIVE'
ORDER BY premium_amount DESC

In [ ]:
# Exercise 2.2 - Filter with isin + select specific columns + order

target_reinsurers = ["SWISS-RE-001", "MUNICH-RE-001", "HANNOVER-RE-001"]

# Filter: col("reinsurer_id").isin(target_reinsurers) & status == "ACTIVE"
# Select: "contract_id", "reinsurer_id", "premium_amount", "currency", "status"
# Order by: premium_amount descending
selected_reinsurers = (
    contracts_df
    .filter(________)
    .select(________)
    .orderBy(________)
)

print(f"Selected reinsurer contracts: {selected_reinsurers.count()}")
display(selected_reinsurers.limit(10))

In [ ]:
# Verification 2.2
assert selected_reinsurers.count() > 0, "No results - check isin() and status filter"
assert len(selected_reinsurers.columns) == 5, f"Expected 5 columns, got {len(selected_reinsurers.columns)} - check .select()"
reinsurer_ids = [r.reinsurer_id for r in selected_reinsurers.select("reinsurer_id").distinct().collect()]
assert set(reinsurer_ids).issubset({"SWISS-RE-001", "MUNICH-RE-001", "HANNOVER-RE-001"}), f"Unexpected reinsurer_ids: {reinsurer_ids}"
statuses = [r.status for r in selected_reinsurers.select("status").distinct().collect()]
assert statuses == ["ACTIVE"], f"Expected only ACTIVE, got {statuses}"
print(f"Exercise 2.2 passed! {selected_reinsurers.count()} contracts from target reinsurers")

In [1]:
displayHTML("""
<details style="margin: 10px 0;">
  <summary style="cursor: pointer; color: #1B3A4B; font-weight: bold; padding: 8px 0;">Click to reveal answer</summary>
  <div style="background: #F9F7F4; padding: 12px 16px; border-radius: 8px; margin-top: 8px; border-left: 4px solid #F9A825;">
    <pre style="margin: 8px 0; color: #1B1F24; background: #FFFFFF; padding: 10px; border-radius: 6px;">
selected_reinsurers = (
    contracts_df
    .filter(
        (col("reinsurer_id").isin(target_reinsurers)) &amp;
        (col("status") == "ACTIVE")
    )
    .select(
        "contract_id", "reinsurer_id", "premium_amount", "currency", "status"
    )
    .orderBy(desc("premium_amount"))
)
    </pre>
  </div>
</details>
""")

Click to reveal answer 
 
 
selected_reinsurers = (
 contracts_df
 .filter(
 (col("reinsurer_id").isin(target_reinsurers)) &
 (col("status") == "ACTIVE")
 )
 .select(
 "contract_id", "reinsurer_id", "premium_amount", "currency", "status"
 )
 .orderBy(desc("premium_amount"))
)

In [1]:
displayHTML("""
<div style="background-color: #F1F8E9; padding: 14px 18px; border-radius: 10px; margin: 10px 0; border-left: 4px solid #00A972;">
<strong style="color: #1B1F24;">Key Takeaway:</strong> <span style="color: #1B1F24;">PySpark filtering uses <code>.filter()</code> with <code>col()</code> references. Combine conditions with <code>&amp;</code> (AND) and <code>|</code> (OR), always wrapping each condition in parentheses. Use <code>.isin()</code> for SQL <code>IN</code> clauses.</span>
</div>
""")

Key Takeaway: PySpark filtering uses .filter() with col() references. Combine conditions with & (AND) and | (OR), always wrapping each condition in parentheses. Use .isin() for SQL IN clauses.

---
## Section 3: Adding Columns / Transformations

Adding new columns to a DataFrame is done with `.withColumn()`. This is the PySpark equivalent of `SELECT *, expression AS new_col` in SQL.

Common transformation patterns:
- **Conditional columns**: `when().otherwise()` replacing SQL `CASE WHEN`
- **Date extraction**: Functions like `year()`, `month()`, `date_format()`

### Exercise 3.1 - Conditional Column (CASE WHEN)

Categorize contracts by premium size:
- **HIGH**: premium_amount >= 10,000,000
- **MEDIUM**: premium_amount >= 1,000,000
- **LOW**: everything else

**Fill in the blanks** to build the conditional chain.

In [ ]:
%sql
SELECT *,
  CASE
    WHEN premium_amount >= 10000000 THEN 'HIGH'
    WHEN premium_amount >= 1000000 THEN 'MEDIUM'
    ELSE 'LOW'
  END AS premium_category
FROM contracts

In [ ]:
# Exercise 3.1 - Add a premium_category column using when/otherwise
# HIGH >= 10M, MEDIUM >= 1M, LOW = else

contracts_categorized = contracts_df.withColumn(
    "premium_category",
    ________(col("premium_amount") >= 10000000, "HIGH")    # Hint: first when()
    .________(col("premium_amount") >= 1000000, "MEDIUM")  # Hint: chained when()
    .________("LOW")                                        # Hint: default value
)

contracts_categorized.select(
    "contract_id", "premium_amount", "premium_category"
).limit(10))

In [ ]:
# Verification 3.1
assert "premium_category" in contracts_categorized.columns, "Column premium_category not found"
categories = [r.premium_category for r in contracts_categorized.select("premium_category").distinct().collect()]
assert set(categories) == {"HIGH", "MEDIUM", "LOW"}, f"Expected HIGH/MEDIUM/LOW, got {categories}"
print(f"Exercise 3.1 passed! Categories: {sorted(categories)}")

In [1]:
displayHTML("""
<details style="margin: 10px 0;">
  <summary style="cursor: pointer; color: #1B3A4B; font-weight: bold; padding: 8px 0;">Click to reveal answer</summary>
  <div style="background: #F9F7F4; padding: 12px 16px; border-radius: 8px; margin-top: 8px; border-left: 4px solid #F9A825;">
    <pre style="margin: 8px 0; color: #1B1F24; background: #FFFFFF; padding: 10px; border-radius: 6px;">
contracts_categorized = contracts_df.withColumn(
    "premium_category",
    when(col("premium_amount") &gt;= 10000000, "HIGH")
    .when(col("premium_amount") &gt;= 1000000, "MEDIUM")
    .otherwise("LOW")
)
    </pre>
  </div>
</details>
""")

Click to reveal answer 
 
 
contracts_categorized = contracts_df.withColumn(
 "premium_category",
 when(col("premium_amount") >= 10000000, "HIGH")
 .when(col("premium_amount") >= 1000000, "MEDIUM")
 .otherwise("LOW")
)

### Exercise 3.2 - Date Extraction

Extract the year from the `effective_date` column to analyze contracts by year.

**Fill in the blank** with the correct date function.

In [ ]:
%sql
SELECT *, YEAR(effective_date) AS effective_year
FROM contracts

In [ ]:
# Exercise 3.2 - Extract year from effective_date

contracts_with_year = contracts_df.withColumn(
    "effective_year",
    ________(col("effective_date"))   # Hint: function to extract year from a date
)

contracts_with_year.select(
    "contract_id", "effective_date", "effective_year"
).limit(5))

In [ ]:
# Verification 3.2
assert "effective_year" in contracts_with_year.columns, "Column effective_year not found"
sample_year = contracts_with_year.select("effective_year").first()[0]
assert 2020 <= sample_year <= 2030, f"Year {sample_year} looks wrong - check the year() function"
print(f"Exercise 3.2 passed! Sample year: {sample_year}")

In [1]:
displayHTML("""
<details style="margin: 10px 0;">
  <summary style="cursor: pointer; color: #1B3A4B; font-weight: bold; padding: 8px 0;">Click to reveal answer</summary>
  <div style="background: #F9F7F4; padding: 12px 16px; border-radius: 8px; margin-top: 8px; border-left: 4px solid #F9A825;">
    <pre style="margin: 8px 0; color: #1B1F24; background: #FFFFFF; padding: 10px; border-radius: 6px;">
contracts_with_year = contracts_df.withColumn(
    "effective_year",
    year(col("effective_date"))
)
    </pre>
  </div>
</details>
""")

Click to reveal answer 
 
 
contracts_with_year = contracts_df.withColumn(
 "effective_year",
 year(col("effective_date"))
)

In [1]:
displayHTML("""
<div style="background-color: #F1F8E9; padding: 14px 18px; border-radius: 10px; margin: 10px 0; border-left: 4px solid #00A972;">
<strong style="color: #1B1F24;">Key Takeaway:</strong> <span style="color: #1B1F24;">Use <code>.withColumn()</code> to add or replace columns. Conditional logic uses <code>when().when().otherwise()</code> chains. Date functions like <code>year()</code> and <code>month()</code> extract components from date columns.</span>
</div>
""")

Key Takeaway: Use .withColumn() to add or replace columns. Conditional logic uses when().when().otherwise() chains. Date functions like year() and month() extract components from date columns.

---
## Section 4: Joining DataFrames

Joining is how we combine data from multiple tables (DataFrames). In PySpark, the `.join()` method takes three arguments:
1. The other DataFrame
2. The join condition
3. The join type ("inner", "left", "right", "full")

In [1]:
displayHTML("""
<div style="background-color: #F9F7F4; padding: 20px; border-radius: 12px; border: 1px solid #E8E5E0; margin: 10px 0; box-shadow: 0 2px 8px rgba(0,0,0,0.06);">

<h4 style="margin-top:0; color: #1B3A4B;">Data Model</h4>

<div style="display: flex; gap: 30px; align-items: flex-start; flex-wrap: wrap;">

  <div style="flex: 1; min-width: 250px;">
    <table style="width: 100%; border-collapse: collapse; font-size: 13px;">
      <tr style="background: #1B3A4B; color: #FFF;"><th colspan="2" style="padding: 10px; text-align: left; border-radius: 6px 6px 0 0;">contracts</th></tr>
      <tr style="background: #FFF;"><td style="padding: 6px 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><strong>contract_id</strong></td><td style="padding: 6px 10px; border-bottom: 1px solid #E8E5E0; color: #888;">PK</td></tr>
      <tr style="background: #F9F7F4;"><td style="padding: 6px 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><strong>axa_entity_id</strong></td><td style="padding: 6px 10px; border-bottom: 1px solid #E8E5E0; color: #FF5F46;">FK &rarr; entities</td></tr>
      <tr style="background: #FFF;"><td style="padding: 6px 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">reinsurer_id</td><td style="padding: 6px 10px; border-bottom: 1px solid #E8E5E0; color: #888;"></td></tr>
      <tr style="background: #F9F7F4;"><td style="padding: 6px 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">contract_type</td><td style="padding: 6px 10px; border-bottom: 1px solid #E8E5E0; color: #888;"></td></tr>
      <tr style="background: #FFF;"><td style="padding: 6px 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">effective_date</td><td style="padding: 6px 10px; border-bottom: 1px solid #E8E5E0; color: #888;"></td></tr>
      <tr style="background: #F9F7F4;"><td style="padding: 6px 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">expiry_date</td><td style="padding: 6px 10px; border-bottom: 1px solid #E8E5E0; color: #888;"></td></tr>
      <tr style="background: #FFF;"><td style="padding: 6px 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">premium_amount</td><td style="padding: 6px 10px; border-bottom: 1px solid #E8E5E0; color: #888;">DOUBLE</td></tr>
      <tr style="background: #F9F7F4;"><td style="padding: 6px 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">cession_rate</td><td style="padding: 6px 10px; border-bottom: 1px solid #E8E5E0; color: #888;">DOUBLE</td></tr>
      <tr style="background: #FFF;"><td style="padding: 6px 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">currency</td><td style="padding: 6px 10px; border-bottom: 1px solid #E8E5E0; color: #888;"></td></tr>
      <tr style="background: #F9F7F4;"><td style="padding: 6px 10px; color: #1B1F24;">status</td><td style="padding: 6px 10px; color: #888;"></td></tr>
    </table>
  </div>

  <div style="flex: 1; min-width: 280px;">
    <table style="width: 100%; border-collapse: collapse; font-size: 13px;">
      <tr style="background: #1B3A4B; color: #FFF;"><th colspan="2" style="padding: 10px; text-align: left; border-radius: 6px 6px 0 0;">entities</th></tr>
      <tr style="background: #FFF;"><td style="padding: 6px 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><strong>entity_id</strong></td><td style="padding: 6px 10px; border-bottom: 1px solid #E8E5E0; color: #888;">PK</td></tr>
      <tr style="background: #F9F7F4;"><td style="padding: 6px 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">entity_name</td><td style="padding: 6px 10px; border-bottom: 1px solid #E8E5E0; color: #888;"></td></tr>
      <tr style="background: #FFF;"><td style="padding: 6px 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">entity_type</td><td style="padding: 6px 10px; border-bottom: 1px solid #E8E5E0; color: #888;"></td></tr>
      <tr style="background: #F9F7F4;"><td style="padding: 6px 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">region</td><td style="padding: 6px 10px; border-bottom: 1px solid #E8E5E0; color: #888;"></td></tr>
      <tr style="background: #FFF;"><td style="padding: 6px 10px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">country</td><td style="padding: 6px 10px; border-bottom: 1px solid #E8E5E0; color: #888;"></td></tr>
      <tr style="background: #F9F7F4;"><td style="padding: 6px 10px; border-bottom: 1px solid #E8E5E0; color: #FF5F46;"><strong>address</strong> (STRUCT)</td><td style="padding: 6px 10px; border-bottom: 1px solid #E8E5E0; color: #888;"></td></tr>
      <tr style="background: #FFF;"><td style="padding: 6px 10px 6px 30px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">.city</td><td style="padding: 6px 10px; border-bottom: 1px solid #E8E5E0; color: #888;">STRING</td></tr>
      <tr style="background: #F9F7F4;"><td style="padding: 6px 10px 6px 30px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">.postal_code</td><td style="padding: 6px 10px; border-bottom: 1px solid #E8E5E0; color: #888;">STRING</td></tr>
      <tr style="background: #FFF;"><td style="padding: 6px 10px; border-bottom: 1px solid #E8E5E0; color: #FF5F46;"><strong>contact</strong> (STRUCT)</td><td style="padding: 6px 10px; border-bottom: 1px solid #E8E5E0; color: #888;"></td></tr>
      <tr style="background: #F9F7F4;"><td style="padding: 6px 10px 6px 30px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">.email</td><td style="padding: 6px 10px; border-bottom: 1px solid #E8E5E0; color: #888;">STRING</td></tr>
      <tr style="background: #FFF;"><td style="padding: 6px 10px 6px 30px; color: #1B1F24;">.phone</td><td style="padding: 6px 10px; color: #888;">STRING</td></tr>
    </table>
  </div>

</div>

<p style="margin-top: 12px; color: #1B3A4B; font-size: 13px;"><strong>Join ON:</strong> <code>contracts.axa_entity_id = entities.entity_id</code></p>

</div>
""")

Data Model 

 

 
 
 contracts 
 contract_id PK 
 axa_entity_id FK → entities 
 reinsurer_id 
 contract_type 
 effective_date 
 expiry_date 
 premium_amount DOUBLE 
 cession_rate DOUBLE 
 currency 
 status 
 
 

 
 
 entities 
 entity_id PK 
 entity_name 
 entity_type 
 region 
 country 
 address (STRUCT) 
 .city STRING 
 .postal_code STRING 
 contact (STRUCT) 
 .email STRING 
 .phone STRING 
 
 

 

 Join ON: contracts.axa_entity_id = entities.entity_id

### Exercise 4.1 - Inner Join

Join contracts with entities to enrich contract data with entity details (name, region, country).

**Fill in the blanks** with the join condition and join type.

In [ ]:
%sql
SELECT *
FROM contracts c
INNER JOIN entities e ON c.axa_entity_id = e.entity_id

In [ ]:
# Exercise 4.1 - Inner join contracts with entities

joined_df = contracts_df.join(
    entities_df,
    contracts_df["axa_entity_id"] ________ entities_df["entity_id"],   # Hint: equality operator
    "________"                                                          # Hint: join type
)

print(f"Joined records: {joined_df.count()}")
display(joined_df.limit(5))

In [ ]:
# Verification 4.1
assert joined_df.count() > 0, "Join produced no results - check condition and join type"
assert "entity_name" in joined_df.columns, "entity_name not in result - join may have failed"
print(f"Exercise 4.1 passed! {joined_df.count()} joined records")

In [1]:
displayHTML("""
<details style="margin: 10px 0;">
  <summary style="cursor: pointer; color: #1B3A4B; font-weight: bold; padding: 8px 0;">Click to reveal answer</summary>
  <div style="background: #F9F7F4; padding: 12px 16px; border-radius: 8px; margin-top: 8px; border-left: 4px solid #F9A825;">
    <pre style="margin: 8px 0; color: #1B1F24; background: #FFFFFF; padding: 10px; border-radius: 6px;">
joined_df = contracts_df.join(
    entities_df,
    contracts_df["axa_entity_id"] == entities_df["entity_id"],
    "inner"
)
    </pre>
  </div>
</details>
""")

Click to reveal answer 
 
 
joined_df = contracts_df.join(
 entities_df,
 contracts_df["axa_entity_id"] == entities_df["entity_id"],
 "inner"
)

### Exercise 4.2 - Select Specific Columns After Join

After a join, you often have duplicate or unnecessary columns. Select only what you need, including nested struct fields with **dot notation**.

**Fill in the blanks** to select 5 columns: `contract_id`, `entity_name`, `contract_type`, `premium_amount`, and `city` (from `address.city`).

In [ ]:
# Exercise 4.2 - Select specific columns after join to avoid ambiguity
# Select: contract_id, entity_name, contract_type, premium_amount, city (from address.city)

enriched_contracts = joined_df.select(
    contracts_df[________],
    entities_df[________],
    contracts_df[________],
    contracts_df[________],
    entities_df[________].alias("city"),
)

# Register for SQL usage in later sections
enriched_contracts.createOrReplaceTempView("enriched_contracts")

print(f"Enriched contracts: {enriched_contracts.count()} rows, {len(enriched_contracts.columns)} columns")
display(enriched_contracts.limit(5))

In [1]:
displayHTML("""
<div style="background-color: #F1F8E9; padding: 14px 18px; border-radius: 10px; margin: 10px 0; border-left: 4px solid #00A972;">
<strong style="color: #1B1F24;">Key Takeaway:</strong> <span style="color: #1B1F24;">PySpark joins use <code>df1.join(df2, condition, type)</code>. Always select specific columns after a join to avoid ambiguous column references. Use <code>df["col"]</code> syntax to disambiguate columns with the same name from different DataFrames.</span>
</div>
""")

Key Takeaway: PySpark joins use df1.join(df2, condition, type) . Always select specific columns after a join to avoid ambiguous column references. Use df["col"] syntax to disambiguate columns with the same name from different DataFrames.

---
## Section 5: Aggregations

Aggregation is how we summarize data. The PySpark pattern is:

```python
df.groupBy("col1", "col2").agg(
    function("col").alias("name"),
    ...
)
```

This is equivalent to SQL `GROUP BY` with aggregate functions.

### Exercise 5.1 - Aggregate by Region

Calculate the number of contracts, total premium, and average premium per region.

**Fill in the blanks** with the correct aggregate functions.

In [ ]:
%sql
SELECT region,
       COUNT(*) AS contract_count,
       SUM(premium_amount) AS total_premium,
       AVG(premium_amount) AS avg_premium
FROM enriched_contracts
GROUP BY region
ORDER BY total_premium DESC

In [ ]:
# Exercise 5.1 - Aggregate by region

region_summary = (
    enriched_contracts
    .groupBy("region")
    .agg(
        count("contract_id").alias("contract_count"),
        ________("premium_amount").alias("total_premium"),   # Hint: F.sum
        ________("premium_amount").alias("avg_premium")      # Hint: average function
    )
    .orderBy(desc("total_premium"))
)

display(region_summary)

In [ ]:
# Verification 5.1
assert region_summary.count() > 0, "No results - check groupBy column"
cols = region_summary.columns
assert "total_premium" in cols and "avg_premium" in cols, "Missing aggregation columns"
print(f"Exercise 5.1 passed! {region_summary.count()} regions summarized")

In [1]:
displayHTML("""
<details style="margin: 10px 0;">
  <summary style="cursor: pointer; color: #1B3A4B; font-weight: bold; padding: 8px 0;">Click to reveal answer</summary>
  <div style="background: #F9F7F4; padding: 12px 16px; border-radius: 8px; margin-top: 8px; border-left: 4px solid #F9A825;">
    <pre style="margin: 8px 0; color: #1B1F24; background: #FFFFFF; padding: 10px; border-radius: 6px;">
First blank: <code>F.sum</code> -- aliased import to avoid conflict with Python's built-in sum()
Second blank: <code>avg</code> -- average function, same name as SQL
    </pre>
  </div>
</details>
""")

Click to reveal answer 
 
 
First blank: F.sum -- aliased import to avoid conflict with Python's built-in sum()
Second blank: avg -- average function, same name as SQL

### Exercise 5.2 - Aggregate by Multiple Columns

Break down the analysis further by grouping on both region and contract type.

**Fill in the blanks** with the correct column names in `groupBy()`.

In [ ]:
%sql
SELECT region, contract_type,
       COUNT(*) AS contract_count,
       SUM(premium_amount) AS total_premium,
       ROUND(AVG(premium_amount), 2) AS avg_premium
FROM enriched_contracts
GROUP BY region, contract_type
ORDER BY region, total_premium DESC

In [ ]:
# Exercise 5.2 - Aggregate by region AND contract_type

region_type_summary = (
    enriched_contracts
    .groupBy("________", "________")   # Hint: two grouping columns
    .agg(
        count("contract_id").alias("contract_count"),
        F.sum("premium_amount").alias("total_premium"),
        F.round(avg("premium_amount"), 2).alias("avg_premium")
    )
    .orderBy("region", desc("total_premium"))
)

display(region_type_summary.limit(20))

In [ ]:
# Verification 5.2
assert region_type_summary.count() > region_summary.count(), "Should have more rows than region-only summary"
print(f"Exercise 5.2 passed! {region_type_summary.count()} region+type combinations")

In [1]:
displayHTML("""
<details style="margin: 10px 0;">
  <summary style="cursor: pointer; color: #1B3A4B; font-weight: bold; padding: 8px 0;">Click to reveal answer</summary>
  <div style="background: #F9F7F4; padding: 12px 16px; border-radius: 8px; margin-top: 8px; border-left: 4px solid #F9A825;">
    <pre style="margin: 8px 0; color: #1B1F24; background: #FFFFFF; padding: 10px; border-radius: 6px;">
First blank: <code>"region"</code>
Second blank: <code>"contract_type"</code>
    </pre>
  </div>
</details>
""")

Click to reveal answer 
 
 
First blank: "region" 
Second blank: "contract_type"

In [1]:
displayHTML("""
<div style="background-color: #F1F8E9; padding: 14px 18px; border-radius: 10px; margin: 10px 0; border-left: 4px solid #00A972;">
<strong style="color: #1B1F24;">Key Takeaway:</strong> <span style="color: #1B1F24;">PySpark aggregations use <code>.groupBy().agg()</code>. Multiple aggregate functions go inside <code>.agg()</code> separated by commas. Use aliased imports (<code>F.sum</code>, <code>F.max</code>) to avoid conflicts with Python built-ins. Always <code>.alias()</code> your results for clarity.</span>
</div>
""")

Key Takeaway: PySpark aggregations use .groupBy().agg() . Multiple aggregate functions go inside .agg() separated by commas. Use aliased imports ( F.sum , F.max ) to avoid conflicts with Python built-ins. Always .alias() your results for clarity.

---
## Section 6: Complex Transformations - Window Functions

Window functions are one of the most powerful features in both SQL and PySpark. They let you perform calculations **across a set of rows related to the current row**, without collapsing the rows like `GROUP BY` does.

In [1]:
displayHTML("""
<div style="background-color: #F9F7F4; padding: 20px; border-radius: 12px; border: 1px solid #E8E5E0; margin: 10px 0; box-shadow: 0 2px 8px rgba(0,0,0,0.06);">

<h4 style="margin-top: 0; color: #1B3A4B;">Window Functions Explained</h4>

<p style="color: #1B1F24;"><strong>GROUP BY vs Window Functions:</strong></p>
<ul style="color: #1B1F24;">
  <li><code>GROUP BY</code> collapses rows into summary rows (N rows become fewer rows)</li>
  <li><strong>Window functions</strong> add a calculated column to each row (N rows stay N rows)</li>
</ul>

<pre style="font-family: monospace; font-size: 13px; background-color: #FFFFFF; padding: 15px; border-radius: 8px; color: #1B1F24; border: 1px solid #E8E5E0;">
                      Window = PARTITION BY entity + ORDER BY premium

  entity_id | premium  | rank       Think of it as:
  ----------|----------|------      - PARTITION BY = "group" rows into windows
  ENT_001   | 10,000   |  1         - ORDER BY = sort within each window
  ENT_001   |  8,000   |  2         - Function = compute over the window
  ENT_001   |  5,000   |  3              (rank, sum, lag, lead, etc.)
  ----------|----------|------
  ENT_002   | 12,000   |  1         Each partition is processed independently.
  ENT_002   |  7,000   |  2         Original rows are preserved!
  ENT_002   |  3,000   |  3
</pre>

<p style="color: #1B1F24;"><strong>PySpark syntax:</strong></p>
<pre style="font-family: monospace; font-size: 13px; background-color: #FFFFFF; padding: 15px; border-radius: 8px; color: #1B1F24; border: 1px solid #E8E5E0;">
# Step 1: Define the window specification
window_spec = Window.partitionBy("partition_col").orderBy(desc("order_col"))

# Step 2: Apply a window function over it
df.withColumn("result", dense_rank().over(window_spec))
</pre>

</div>
""")

Window Functions Explained 

 GROUP BY vs Window Functions: 
 
 GROUP BY collapses rows into summary rows (N rows become fewer rows) 
 Window functions add a calculated column to each row (N rows stay N rows) 
 

 
 Window = PARTITION BY entity + ORDER BY premium

 entity_id | premium | rank Think of it as:
 ----------|----------|------ - PARTITION BY = "group" rows into windows
 ENT_001 | 10,000 | 1 - ORDER BY = sort within each window
 ENT_001 | 8,000 | 2 - Function = compute over the window
 ENT_001 | 5,000 | 3 (rank, sum, lag, lead, etc.)
 ----------|----------|------
 ENT_002 | 12,000 | 1 Each partition is processed independently.
 ENT_002 | 7,000 | 2 Original rows are preserved!
 ENT_002 | 3,000 | 3
 

 PySpark syntax: 
 
# Step 1: Define the window specification
window_spec = Window.partitionBy("partition_col").orderBy(desc("order_col"))

# Step 2: Apply a window function over it
df.withColumn("result", dense_rank().over(window_spec))

### Exercise 6.1 - Rank Contracts by Premium Within Each Entity

For each AXA entity, rank their contracts from highest to lowest premium using `dense_rank()`. This helps identify the most valuable contracts per entity.

**Fill in the blanks** to define the window and apply the ranking function.

In [ ]:
%sql
SELECT *,
    DENSE_RANK() OVER (
        PARTITION BY axa_entity_id
        ORDER BY premium_amount DESC
    ) AS premium_rank
FROM enriched_contracts

In [ ]:
# Exercise 6.1 - Rank contracts by premium within each entity

# Step 1: Define the window specification
entity_premium_window = Window.________("axa_entity_id").orderBy(desc("premium_amount"))   # Hint: partition by entity

# Step 2: Apply dense_rank over the window
ranked_contracts = enriched_contracts.withColumn(
    "premium_rank",
    ________().over(entity_premium_window)   # Hint: ranking function that handles ties without gaps
)

# Show top 3 contracts per entity
ranked_contracts.filter(col("premium_rank") <= 3).orderBy(
    "axa_entity_id", "premium_rank"
).select(
    "axa_entity_id", "entity_name", "contract_id", "premium_amount", "premium_rank"
).limit(15))

In [ ]:
# Verification 6.1
assert "premium_rank" in ranked_contracts.columns, "premium_rank column not found"
top1 = ranked_contracts.filter(col("premium_rank") == 1)
assert top1.count() > 0, "No rank-1 records found"
print(f"Exercise 6.1 passed! {top1.count()} entities have a top-ranked contract")

In [1]:
displayHTML("""
<details style="margin: 10px 0;">
  <summary style="cursor: pointer; color: #1B3A4B; font-weight: bold; padding: 8px 0;">Click to reveal answer</summary>
  <div style="background: #F9F7F4; padding: 12px 16px; border-radius: 8px; margin-top: 8px; border-left: 4px solid #F9A825;">
    <pre style="margin: 8px 0; color: #1B1F24; background: #FFFFFF; padding: 10px; border-radius: 6px;">
entity_premium_window = Window.partitionBy("axa_entity_id").orderBy(desc("premium_amount"))

ranked_contracts = enriched_contracts.withColumn(
    "premium_rank",
    dense_rank().over(entity_premium_window)
)
    </pre>
  </div>
</details>
""")

Click to reveal answer 
 
 
entity_premium_window = Window.partitionBy("axa_entity_id").orderBy(desc("premium_amount"))

ranked_contracts = enriched_contracts.withColumn(
 "premium_rank",
 dense_rank().over(entity_premium_window)
)

### Exercise 6.2 - Running Total of Premiums by Entity

Calculate a cumulative (running) total of premiums for each entity, ordered by effective date. This shows how premium exposure builds up over time.

**Fill in the blanks** to define the window and apply the running sum.

In [ ]:
%sql
SELECT *,
    SUM(premium_amount) OVER (
        PARTITION BY axa_entity_id
        ORDER BY effective_date
    ) AS running_total
FROM enriched_contracts

In [ ]:
# Exercise 6.2 - Running total of premiums by entity

# Define window: partition by entity, order by date (ascending for chronological running total)
entity_date_window = Window.partitionBy("________").orderBy("________")   # Hint: partition by entity, order by date

running_totals = enriched_contracts.withColumn(
    "running_total",
    ________("premium_amount").over(entity_date_window)   # Hint: cumulative sum function (aliased import)
)

running_totals.select(
    "axa_entity_id", "entity_name", "contract_id", "effective_date", "premium_amount", "running_total"
).orderBy("axa_entity_id", "effective_date").limit(15))

In [ ]:
# Verification 6.2
assert "running_total" in running_totals.columns, "running_total column not found"
sample_rt = running_totals.select("running_total").first()[0]
assert sample_rt > 0, "Running total should be positive"
print(f"Exercise 6.2 passed! Running totals calculated")

In [1]:
displayHTML("""
<details style="margin: 10px 0;">
  <summary style="cursor: pointer; color: #1B3A4B; font-weight: bold; padding: 8px 0;">Click to reveal answer</summary>
  <div style="background: #F9F7F4; padding: 12px 16px; border-radius: 8px; margin-top: 8px; border-left: 4px solid #F9A825;">
    <pre style="margin: 8px 0; color: #1B1F24; background: #FFFFFF; padding: 10px; border-radius: 6px;">
entity_date_window = Window.partitionBy("axa_entity_id").orderBy("effective_date")

running_totals = enriched_contracts.withColumn(
    "running_total",
    F.sum("premium_amount").over(entity_date_window)
)
    </pre>
  </div>
</details>
""")

Click to reveal answer 
 
 
entity_date_window = Window.partitionBy("axa_entity_id").orderBy("effective_date")

running_totals = enriched_contracts.withColumn(
 "running_total",
 F.sum("premium_amount").over(entity_date_window)
)

### Exercise 6.3 - Compare with Previous and Next Contract (Lag / Lead)

`lag()` looks at the **previous** row and `lead()` looks at the **next** row within the window. This is useful for comparing each contract's premium with its neighbors.

**Fill in the blanks** using `lag()` and `lead()`.

In [ ]:
%sql
SELECT *,
    LAG(premium_amount, 1) OVER (
        PARTITION BY axa_entity_id ORDER BY effective_date
    ) AS prev_premium,
    LEAD(premium_amount, 1) OVER (
        PARTITION BY axa_entity_id ORDER BY effective_date
    ) AS next_premium
FROM enriched_contracts

In [ ]:
# Exercise 6.3 - Lag and Lead: compare with previous/next contract premium

# Reuse the entity_date_window from Exercise 6.2
# (partitioned by axa_entity_id, ordered by effective_date)

lag_lead_df = (
    enriched_contracts
    .withColumn(
        "prev_premium",
        ________("premium_amount", 1).over(entity_date_window)   # Hint: look at previous row
    )
    .withColumn(
        "next_premium",
        ________("premium_amount", 1).over(entity_date_window)   # Hint: look at next row
    )
    .withColumn(
        "premium_change",
        col("premium_amount") - col("prev_premium")
    )
)

lag_lead_df.select(
    "axa_entity_id", "contract_id", "effective_date",
    "prev_premium", "premium_amount", "next_premium", "premium_change"
).orderBy("axa_entity_id", "effective_date").limit(15))

In [ ]:
# Verification 6.3
assert "prev_premium" in lag_lead_df.columns, "prev_premium not found - check lag()"
assert "next_premium" in lag_lead_df.columns, "next_premium not found - check lead()"
print(f"Exercise 6.3 passed! Lag/lead columns added")

In [1]:
displayHTML("""
<details style="margin: 10px 0;">
  <summary style="cursor: pointer; color: #1B3A4B; font-weight: bold; padding: 8px 0;">Click to reveal answer</summary>
  <div style="background: #F9F7F4; padding: 12px 16px; border-radius: 8px; margin-top: 8px; border-left: 4px solid #F9A825;">
    <pre style="margin: 8px 0; color: #1B1F24; background: #FFFFFF; padding: 10px; border-radius: 6px;">
First blank: <code>lag</code> -- looks at the previous row
Second blank: <code>lead</code> -- looks at the next row
    </pre>
  </div>
</details>
""")

Click to reveal answer 
 
 
First blank: lag -- looks at the previous row
Second blank: lead -- looks at the next row

---
## Section 7: Array Functions

### Exercise 7.1 - Array Functions: explode, array_contains, array_distinct

Array functions let you work with columns that contain lists of values. Common patterns:
- **`split()`** — Convert a string to an array
- **`explode()`** — One row per array element (like LATERAL FLATTEN in SQL)
- **`array_contains()`** — Check if an array contains a value
- **`array_distinct()`** — Remove duplicates from an array
- **`collect_list()`** — Aggregate values into an array (reverse of explode)

In this exercise, we will create arrays from our contracts data and manipulate them.

In [ ]:
# Exercise 7.1a - collect_list + array_distinct
# Group contracts by entity to get a list of contract types per entity
entity_contracts = enriched_contracts.groupBy("axa_entity_id", "entity_name").agg(
    F.collect_list("contract_type").alias("contract_types_raw")
)

# Remove duplicate contract types from the array
entity_contracts = entity_contracts.withColumn(
    "contract_types",
    ________(col("contract_types_raw"))
)

print("Contract types per entity:")
entity_contracts.select("entity_name", "contract_types"))

In [ ]:
# Exercise 7.1b - array_contains
# Keep only entities that have QUOTA_SHARE in their contract types
quota_share_entities = entity_contracts.filter(
    ________(col("contract_types"), "QUOTA_SHARE")
)

print(f"Entities offering QUOTA_SHARE: {quota_share_entities.count()}")
quota_share_entities.select("entity_name", "contract_types"))

In [ ]:
# Exercise 7.1c - explode
# Create one row per contract type per entity
exploded_df = entity_contracts.select(
    col("entity_name"),
    ________(col("contract_types")).alias("contract_type")
)

print("Exploded contract types:")
display(exploded_df.limit(20))

In [1]:
displayHTML("""
<details style="margin: 10px 0;">
  <summary style="cursor: pointer; color: #1B3A4B; font-weight: bold; padding: 8px 0;">Click to reveal hints</summary>
  <div style="background: #F9F7F4; padding: 12px 16px; border-radius: 8px; margin-top: 8px; border-left: 4px solid #F9A825;">
    <ul style="margin: 8px 0; padding-left: 20px;">
  <li style="margin: 6px 0; color: #1B1F24;">Part A: <code>array_distinct</code> — removes duplicate values from an array column</li>
  <li style="margin: 6px 0; color: #1B1F24;">Part B: <code>array_contains</code> — returns true if the array contains the specified value</li>
  <li style="margin: 6px 0; color: #1B1F24;">Part C: <code>explode</code> — creates one row per array element (like UNNEST in SQL)</li>
</ul>
  </div>
</details>
""")

Click to reveal hints 
 
 
 Part A: array_distinct — removes duplicate values from an array column 
 Part B: array_contains — returns true if the array contains the specified value 
 Part C: explode — creates one row per array element (like UNNEST in SQL)

In [1]:
displayHTML("""
<div style="background-color: #F1F8E9; padding: 14px 18px; border-radius: 10px; margin: 10px 0; border-left: 4px solid #00A972;">
<strong style="color: #1B1F24;">Key Takeaway:</strong> <span style="color: #1B1F24;">Array functions bridge the gap between nested/semi-structured data and flat relational tables. Use <code>collect_list()</code> to aggregate into arrays, <code>array_distinct()</code> to deduplicate, <code>array_contains()</code> to filter, <code>explode()</code> to flatten, and <code>size()</code> to count elements. These are essential for working with JSON, CDC, and multi-valued fields.</span>
</div>
""")

Key Takeaway: Array functions bridge the gap between nested/semi-structured data and flat relational tables. Use collect_list() to aggregate into arrays, array_distinct() to deduplicate, array_contains() to filter, explode() to flatten, and size() to count elements. These are essential for working with JSON, CDC, and multi-valued fields.

---
## Section 9: Save to Parquet

In [ ]:
# Save the enriched contracts to a Parquet file
# format: "parquet"
# mode: "overwrite"
(
    enriched_contracts
    .write
    .format(________)
    .mode(________)
    .save("/Volumes/axa_training/lab_assets/training_volume/output/enriched_contracts.parquet")
)

print(f"Saved to /Volumes/axa_training/lab_assets/training_volume/output/enriched_contracts.parquet")

In [ ]:
# Verify the saved Parquet file
verification_df = spark.read.parquet("/Volumes/axa_training/lab_assets/training_volume/output/enriched_contracts.parquet")
print(f"Rows: {verification_df.count()}, Columns: {len(verification_df.columns)}")
display(verification_df.limit(5))

---
## What You Learned

In this lab, you translated SQL queries into PySpark DataFrame API code, covering the full range of transformations needed for real-world reinsurance analytics.

In [1]:
displayHTML("""
<div style="background-color: #F9F7F4; padding: 20px; border-radius: 12px; border: 1px solid #E8E5E0; margin: 10px 0; box-shadow: 0 2px 8px rgba(0,0,0,0.06);">

<h4 style="margin-top: 0; color: #1B3A4B;">DataFrame API Reference — What You Learned</h4>

<table style="width:100%; border-collapse: collapse; font-size: 13px;">
<tr style="background-color: #1B3A4B; color: #FFFFFF;">
  <th style="padding: 10px; text-align: left; width: 20%;">Category</th>
  <th style="padding: 10px; text-align: left; width: 30%;">Function</th>
  <th style="padding: 10px; text-align: left; width: 50%;">When / How to Use</th>
</tr>
<tr style="background-color: #FFFFFF;">
  <td style="padding: 8px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;" rowspan="3"><strong>Read Data</strong></td>
  <td style="padding: 8px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><code>spark.read.csv(path)</code></td>
  <td style="padding: 8px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">Load CSV files — use <code>.option("header","true")</code> and <code>.option("inferSchema","true")</code></td>
</tr>
<tr style="background-color: #FFFFFF;">
  <td style="padding: 8px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><code>spark.read.schema(s).json(path)</code></td>
  <td style="padding: 8px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">Load JSON with explicit <code>StructType</code> schema — handles nested structs</td>
</tr>
<tr style="background-color: #FFFFFF;">
  <td style="padding: 8px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><code>col("address.city")</code></td>
  <td style="padding: 8px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">Access nested struct fields using dot notation</td>
</tr>
<tr style="background-color: #F9F7F4;">
  <td style="padding: 8px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;" rowspan="2"><strong>Filter</strong></td>
  <td style="padding: 8px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><code>.filter(cond1 &amp; cond2)</code></td>
  <td style="padding: 8px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">Combine conditions with <code>&amp;</code> (AND) or <code>|</code> (OR) — wrap each in parentheses</td>
</tr>
<tr style="background-color: #F9F7F4;">
  <td style="padding: 8px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><code>col("x").isin([...])</code></td>
  <td style="padding: 8px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">Filter where column value is in a list — like SQL <code>IN (...)</code></td>
</tr>
<tr style="background-color: #FFFFFF;">
  <td style="padding: 8px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;" rowspan="2"><strong>Transform</strong></td>
  <td style="padding: 8px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><code>when().when().otherwise()</code></td>
  <td style="padding: 8px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">Conditional column — chain conditions like SQL <code>CASE WHEN</code></td>
</tr>
<tr style="background-color: #FFFFFF;">
  <td style="padding: 8px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><code>year(col("date"))</code></td>
  <td style="padding: 8px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">Extract year/month from date columns</td>
</tr>
<tr style="background-color: #F9F7F4;">
  <td style="padding: 8px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><strong>Join</strong></td>
  <td style="padding: 8px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><code>df1.join(df2, cond, "inner")</code></td>
  <td style="padding: 8px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">Combine DataFrames — use <code>df["col"]</code> syntax to disambiguate after join</td>
</tr>
<tr style="background-color: #FFFFFF;">
  <td style="padding: 8px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><strong>Aggregate</strong></td>
  <td style="padding: 8px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><code>.groupBy().agg(F.sum(), avg())</code></td>
  <td style="padding: 8px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">Group rows and compute summaries — use <code>.alias()</code> to name results</td>
</tr>
<tr style="background-color: #F9F7F4;">
  <td style="padding: 8px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;" rowspan="3"><strong>Window</strong></td>
  <td style="padding: 8px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><code>dense_rank().over(w)</code></td>
  <td style="padding: 8px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">Rank rows within partitions without gaps</td>
</tr>
<tr style="background-color: #F9F7F4;">
  <td style="padding: 8px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><code>F.sum("x").over(w)</code></td>
  <td style="padding: 8px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">Running total — cumulative sum ordered by date</td>
</tr>
<tr style="background-color: #F9F7F4;">
  <td style="padding: 8px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><code>lag() / lead()</code></td>
  <td style="padding: 8px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">Access previous/next row values within a window</td>
</tr>
<tr style="background-color: #FFFFFF;">
  <td style="padding: 8px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;" rowspan="3"><strong>Array</strong></td>
  <td style="padding: 8px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><code>explode(col("arr"))</code></td>
  <td style="padding: 8px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">Flatten array into rows — one row per element</td>
</tr>
<tr style="background-color: #FFFFFF;">
  <td style="padding: 8px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><code>array_contains(col, val)</code></td>
  <td style="padding: 8px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">Check if array column contains a specific value</td>
</tr>
<tr style="background-color: #FFFFFF;">
  <td style="padding: 8px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;"><code>array_distinct(col)</code></td>
  <td style="padding: 8px; border-bottom: 1px solid #E8E5E0; color: #1B1F24;">Remove duplicate values from an array</td>
</tr>
<tr style="background-color: #F9F7F4;">
  <td style="padding: 8px; color: #1B1F24;"><strong>Save</strong></td>
  <td style="padding: 8px; color: #1B1F24;"><code>.write.format("parquet").save(path)</code></td>
  <td style="padding: 8px; color: #1B1F24;">Persist DataFrame as a Parquet file</td>
</tr>
</table>

</div>
""")

DataFrame API Reference — What You Learned 

 
 
 Category 
 Function 
 When / How to Use 
 
 
 Read Data 
 spark.read.csv(path) 
 Load CSV files — use .option("header","true") and .option("inferSchema","true") 
 
 
 spark.read.schema(s).json(path) 
 Load JSON with explicit StructType schema — handles nested structs 
 
 
 col("address.city") 
 Access nested struct fields using dot notation 
 
 
 Filter 
 .filter(cond1 & cond2) 
 Combine conditions with & (AND) or | (OR) — wrap each in parentheses 
 
 
 col("x").isin([...]) 
 Filter where column value is in a list — like SQL IN (...) 
 
 
 Transform 
 when().when().otherwise() 
 Conditional column — chain conditions like SQL CASE WHEN 
 
 
 year(col("date")) 
 Extract year/month from date columns 
 
 
 Join 
 df1.join(df2, cond, "inner") 
 Combine DataFrames — use df["col"] syntax to disambiguate after join 
 
 
 Aggregate 
 .groupBy().agg(F.sum(), avg()) 
 Group rows and compute summaries — use .alias() to name results 
 
 
 Window 
 dense_rank().over(w) 
 Rank rows within partitions without gaps 
 
 
 F.sum("x").over(w) 
 Running total — cumulative sum ordered by date 
 
 
 lag() / lead() 
 Access previous/next row values within a window 
 
 
 Array 
 explode(col("arr")) 
 Flatten array into rows — one row per element 
 
 
 array_contains(col, val) 
 Check if array column contains a specific value 
 
 
 array_distinct(col) 
 Remove duplicate values from an array 
 
 
 Save 
 .write.format("parquet").save(path) 
 Persist DataFrame as a Parquet file

### Key Concepts Covered

- **DataFrame API** is the primary PySpark interface for structured data transformations
- **Window Functions** enable row-level calculations across partitions without collapsing results
- **Chained Transformations** let you build readable, maintainable pipelines step by step
- **Parquet** is the standard columnar format for persisting DataFrames efficiently

### Next Steps

In **Lab 03 - Delta Lake**, you will learn how to:
- Create and manage Delta tables
- Perform MERGE (upsert) operations
- Use time travel to query historical versions of your data
- Optimize Delta tables with Z-ORDER and VACUUM

---
## Cleanup

Remove the temporary views and Delta tables created during this lab.

In [ ]:
# Drop temporary views
spark.catalog.dropTempView("contracts")
spark.catalog.dropTempView("entities")
spark.catalog.dropTempView("enriched_contracts")

print("Temporary views dropped.")

In [ ]:
%sql
-- Cleanup: nothing to drop (saved as Parquet file, not a table)
SELECT 'Cleanup complete' AS status

In [ ]:
print("Lab 02 cleanup complete. All temporary views and tables have been removed.")